# CLEF Flash 4-bit Quantization & Decision Scoring on Tesla T4
**Model:** `Cloudflare/clef-flash` (Qwen3.5-9B multimodal backbone + Joint Schema Head)
**Target:** Google Colab T4 GPU (15-16 GiB VRAM)
**Precision:** 4-bit NormalFloat (NF4) with Double Quantization & FP16 compute
**Task:** Intent-aware adaptive verification & corrective action selection

### Cell 1: Environment & Hardware Diagnostics

In [ ]:
import importlib.metadata as metadata
import platform
import subprocess
import sys
import torch
 
print("=== 1. System & Runtime ===")
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

print("\n=== 2. Core Package Versions ===")
packages = ["torch", "transformers", "accelerate", "bitsandbytes", "huggingface_hub", "safetensors"]
for pkg in packages:
    try:
        print(f"{pkg:16s}: {metadata.version(pkg)}")
    except metadata.PackageNotFoundError:
        print(f"{pkg:16s}: NOT INSTALLED")

print("\n=== 3. PyTorch CUDA Status ===")
print("CUDA Available    :", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("Device Name       :", props.name)
    print(f"Compute Capability: {props.major}.{props.minor}")
    total_mem = props.total_memory / (1024**3)
    free_mem, _ = torch.cuda.mem_get_info(0)
    print(f"Total VRAM        : {total_mem:.2f} GiB")
    print(f"Free VRAM         : {free_mem / (1024**3):.2f} GiB")
    print("FP16 supported    :", True)
    print("BF16 native (T4)  :", torch.cuda.is_bf16_supported(), "(T4 compute 7.5 uses FP16 compute for bitsandbytes)")

print("\n=== 4. nvidia-smi ===")
try:
    smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=True)
    print(smi.stdout)
except Exception as e:
    print("nvidia-smi notice:", e)

### Cell 2: Dependencies & 4-bit Quantization Setup

In [1]:
%pip install -q -U "transformers>=4.48.0" "accelerate>=0.34.0" "bitsandbytes>=0.43.0" safetensors huggingface_hub

import bitsandbytes as bnb
import torch
print("bitsandbytes version:", getattr(bnb, "__version__", "installed"))
print("CUDA available      :", torch.cuda.is_available())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 102.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.9/839.9 kB 59.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.2.0 which is incompatible.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 2.2.0 which is incompatible.
bitsandbytes version: 0.50.2
CUDA available      : True


### Cell 3: Download & Load CLEF Flash with 4-bit NF4 Quantization
Loads the 9B Qwen3.5 backbone in 4-bit NF4 on Tesla T4, reducing VRAM usage from ~18 GiB to ~6.2 GiB.

In [8]:
import gc
from pathlib import Path
import sys
import time
from huggingface_hub import snapshot_download
import torch
from transformers import BitsAndBytesConfig

# Clear GPU memory
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

REPO_ID = "Cloudflare/clef-flash"
LOCAL_DIR = Path("/content/clef_flash_model")

print(f"Downloading model snapshot from {REPO_ID}...")
model_path = Path(
    snapshot_download(
        repo_id=REPO_ID, local_dir=str(LOCAL_DIR), ignore_patterns=["*.bin"]
    )
)
sys.path.insert(0, str(model_path))

import joint_schema_model
from joint_schema_model import load_release_model

print("Configuring 4-bit NF4 Quantization for Tesla T4...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

t0 = time.time()
print("Loading CLEF Flash model via official loader with 4-bit quantization...")
clef_model, processor = load_release_model(
    model_path,
    device="cuda:0",
    dtype=torch.float16,
    quantization_config=bnb_config,
)
load_duration = time.time() - t0

peak_vram = torch.cuda.max_memory_allocated(0) / (1024**3)
free_vram, total_vram = torch.cuda.mem_get_info(0)

print(f"\n✓ CLEF Flash successfully loaded in 4-bit on Tesla T4!")
print(f"Load Time           : {load_duration:.2f} s")
print(f"Peak GPU VRAM       : {peak_vram:.2f} GiB")
print(
    f"Remaining Free VRAM : {free_vram / (1024**3):.2f} GiB /"
    f" {total_vram / (1024**3):.2f} GiB"
)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Configuring 4-bit NF4 Quantization for Tesla T4...
Loading CLEF Flash model via official loader with 4-bit quantization...


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]


✓ CLEF Flash successfully loaded in 4-bit on Tesla T4!
Load Time           : 80.09 s
Peak GPU VRAM       : 7.80 GiB
Remaining Free VRAM : 6.62 GiB / 14.56 GiB


### Cell 4: Decision Scoring Inference (Adaptive Verification Test)

In [10]:
import json
import time
from joint_schema_model import systemone

# SystemOne request specification requires "model", "state", and "questions"
test_payload = {
    "model": "Cloudflare/clef-flash",
    "state": {
        "prompt": (
            "Write a function to return the top-k highest frequency items."
        ),
        "code": "def top_k(arr, k):\n    return sorted(arr)[:k]",
        "execution_result": (
            "FAILED: returns smallest items, not most frequent items"
        ),
    },
    "questions": {
        "has_intent_mismatch": {
            "type": "noul",
            "instructions": (
                "Does the implementation misunderstand user intent regarding"
                " frequency counting?"
            ),
        },
        "recommended_action": {
            "type": "choice",
            "instructions": (
                "Select the optimal corrective action for the code generator."
            ),
            "criteria": {
                "regenerate_full": (
                    "Re-prompt the model from scratch with clarified frequency"
                    " counting intent."
                ),
                "local_patch": (
                    "Apply a surgical patch replacing sorted() with"
                    " Counter.most_common()."
                ),
                "ask_clarification": "Ask the user whether frequency ties matter.",
            },
        },
        "severity_score": {
            "type": "score",
            "instructions": "Rate the severity of the code failure.",
            "criteria": [
                "Minor stylistic flaw",
                "Incorrect edge case",
                "Completely invalid logic",
            ],
        },
    },
}

# Pre-flight check before calling model
assert isinstance(
    test_payload.get("model"), str
), "Error: 'model' key is missing!"
assert "state" in test_payload, "Error: 'state' key is missing!"
assert isinstance(
    test_payload.get("questions"), dict
), "Error: 'questions' key is missing!"

torch.cuda.reset_peak_memory_stats()
t_start = time.time()

decision_output = systemone(
    clef_model, processor, test_payload, max_length=2048
)
latency_ms = (time.time() - t_start) * 1000
peak_infer_vram = torch.cuda.max_memory_allocated(0) / (1024**3)

print(
    f"\n=== Inference Completed in {latency_ms:.2f} ms (Peak VRAM:"
    f" {peak_infer_vram:.2f} GiB) ===\n"
)
print(json.dumps(decision_output, indent=2))


=== Inference Completed in 663.45 ms (Peak VRAM: 7.94 GiB) ===

{
  "model": "Cloudflare/clef-flash",
  "answers": {
    "has_intent_mismatch": {
      "type": "noul",
      "noul": 0.8984
    },
    "recommended_action": {
      "type": "choice",
      "choice": "local_patch",
      "confidence": 0.8496,
      "probabilities": {
        "regenerate_full": 0.1285,
        "local_patch": 0.8496,
        "ask_clarification": 0.022
      }
    },
    "severity_score": {
      "type": "score",
      "score": 1.8284,
      "confidence": 0.8746,
      "legend": {
        "0": "Minor stylistic flaw",
        "1": "Incorrect edge case",
        "2": "Completely invalid logic"
      },
      "probabilities": {
        "0": 0.0461,
        "1": 0.0793,
        "2": 0.8746
      }
    }
  },
  "usage": {
    "input_tokens": 408,
    "output_tokens": 0
  }
}


In [11]:
from google.colab import drive
import json
from pathlib import Path
from datetime import datetime

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define project results path
project_dir = Path("/content/drive/MyDrive/intent_aware_verification_research")
project_dir.mkdir(parents=True, exist_ok=True)

# 3. Save benchmark record
benchmark_log = {
    "timestamp": datetime.now().isoformat(),
    "model": "Cloudflare/clef-flash",
    "quantization": "4-bit NF4 (Double Quant + FP16 compute)",
    "hardware": "NVIDIA Tesla T4 (15.0 GiB)",
    "peak_vram_gib": round(peak_infer_vram, 2),
    "latency_ms": round(latency_ms, 2),
    "test_payload": test_payload,
    "decision_output": decision_output
}

log_file = project_dir / f"clef_flash_benchmark_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
log_file.write_text(json.dumps(benchmark_log, indent=2))

print(f"✓ Experiment benchmark logged to: {log_file}")

Mounted at /content/drive
✓ Experiment benchmark logged to: /content/drive/MyDrive/intent_aware_verification_research/clef_flash_benchmark_20261020_145043.json
